# XRSlab quality-gated analysis

This notebook is the interactive workbench. All computation is delegated to `xrslab.workflow`; the notebook only configures, reviews, approves and exports.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from matplotlib.colors import LogNorm
from matplotlib.patches import Rectangle

from xrsabre.paths import load_workspace
from xrslab.roi_editor import RoiEditor
from xrslab.workflow import (
    AnalysisConfig, QcApproval, build_qc_report, export_analysis,
    finalize_analysis, prepare_analysis,
)

## 1. Configure

This is the only analysis-parameter cell. Paths are resolved at runtime and are not stored in the portable configuration.

In [ ]:
# Analysis configuration
config = AnalysisConfig(
    element='Ho',
    elastic_scan_ids=(57,),
    xrs_scan_ids=(59,),
    analysis_name='XRS_analysis',
    modules=('VB', 'HB', 'HL', 'VD', 'VU'),
    q_range=(0.0, 10.0),
    auto_adjust_rois=True,
    filter_value=0.15,
    elastic_center_range_kev=(9.67, 9.69),
    max_fwhm_ev=2.0,
    min_r_squared=0.8,
    energy_step_ev=0.2,
)
workspace = load_workspace()
print(config)
print('raw:', workspace.raw / config.element)
print('processed:', workspace.processed)

## 2a. Optional interactive ROI editing

Run the next cell when you want to inspect or redraw ROIs before the full pipeline. The editor loads only elastic scan 57. Drag a selected rectangle or its handles to move and resize it; use the controls to add, delete, undo, redo, reset and save. Saving creates versioned files and returns a configuration with further automatic ROI adjustment disabled.

In [ ]:
# Optional: initialise the interactive backend and load only the elastic scan.
from IPython import get_ipython

get_ipython().run_line_magic('matplotlib', 'widget')
roi_editor = RoiEditor.from_config(config, workspace)
roi_editor.display()

In [ ]:
# Run after clicking '另存版本' in the editor. Skip this cell if no edit was saved.
if roi_editor.last_result is not None:
    config = roi_editor.last_result.config
    print('Using manual ROI files:', roi_editor.last_result.filenames)
    print('Automatic ROI adjustment:', config.auto_adjust_rois)
else:
    print('No manual ROI version has been saved; configuration is unchanged.')

## 2. Prepare, calibrate and review QC

This stage validates NeXus inputs, corrects and normalises I0, calibrates ROIs, fits elastic peaks, integrates XRS spectra and builds the QC report. It does not export formal results.

In [ ]:
prepared = prepare_analysis(config, workspace)
qc = build_qc_report(prepared)
display(qc.summary)
display(qc.scan_table)
display(qc.roi_table)

# I0 before/after correction
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for axis, originals, corrected, title in (
    (axes[0], prepared.elastic_i0_original, prepared.elastic_i0_corrected, 'Elastic I0'),
    (axes[1], prepared.xrs_i0_original, prepared.xrs_i0_corrected, 'XRS I0'),
):
    for index, (before, after) in enumerate(zip(originals, corrected)):
        axis.plot(before, alpha=0.4, label=f'{index} raw')
        axis.plot(after, linewidth=1, label=f'{index} corrected')
    axis.set_title(title); axis.set_xlabel('Point'); axis.legend(fontsize=7)
plt.tight_layout()

# Adjusted ROI locations
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for axis, stack, items, title in (
    (axes[0], prepared.elastic_batch.lambda_data[0], prepared.roi_collection.rois, 'lambda'),
    (axes[1], prepared.elastic_batch.minipix_data[0], prepared.roi_collection.minipix_rois, 'minipix'),
):
    image = np.nansum(stack, axis=0)
    positive = image[image > 0]
    norm = LogNorm(vmin=max(float(positive.min()), 1e-12), vmax=float(positive.max())) if positive.size else None
    axis.imshow(image, norm=norm)
    axis.set_title(title)
    for item in items:
        axis.add_patch(Rectangle((item.x1, item.y1), item.x_width, item.y_width, edgecolor='red', facecolor='none', linewidth=0.5))
plt.tight_layout()

# Fit quality and automatically excluded ROI reasons
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(qc.roi_table['r-square'].dropna(), bins=20)
axes[0].axvline(config.min_r_squared, color='red'); axes[0].set_title('R-squared')
axes[1].hist(qc.roi_table['width'].dropna(), bins=20)
axes[1].axvline(config.max_fwhm_ev, color='red'); axes[1].set_title('FWHM (eV)')
plt.tight_layout()
display(qc.roi_table.loc[~qc.roi_table['automatic_accepted'], ['roi_id', 'automatic_exclusion_reasons']])

preview = finalize_analysis(prepared, QcApproval(approved=False))
plt.figure(figsize=(9, 4))
plt.plot(preview.energy_transfer, preview.intensity_sum)
plt.xlabel('Energy Transfer (eV)'); plt.ylabel('Intensity sum'); plt.title('Provisional spectrum')
plt.tight_layout()

## 3. Explicit QC approval

Review the diagnostics above. Add scan IDs or canonical ROI IDs such as `lambda:VU-E1` to the exclusions. Change `approved` to `True` only after review.

In [ ]:
approval = QcApproval(
    approved=False,
    excluded_scans=(),
    excluded_rois=(),
    note='Reviewed ROI, I0, fit and coverage diagnostics',
)
result = finalize_analysis(prepared, approval)
print('approved:', result.approved)
print('selected ROI:', len(result.selected_roi_ids))
print('used XRS scans:', result.used_scan_ids)

## 4. Export immutable run

Formal export is refused unless the approval above is explicit. Each successful export creates a new provenance-rich run directory.

In [ ]:
if approval.approved:
    output_path = export_analysis(
        result, qc, config, workspace,
    )
    print('saved to:', output_path)
else:
    print('QC has not been approved; no formal output was written.')